# 🎯 Python: Decorators

**Advanced Python Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** Higher-order functions, syntactical sugar, aspect-oriented programming
- **Tags:** `python` | `decorators` | `functions` | `citi-prep`

## 📖 The Core Concept in Plain Language

A decorator is a function that takes *another function*, adds some extra functionality to it, and gives it back to you. 

It allows you to explicitly wrap a function with "pre-execution" and "post-execution" steps without ever touching the actual code inside the original function.

### Why This Matters

- **DRY (Don't Repeat Yourself):** If 50 functions all need to log their execution time, you write the timer logic *once* in a decorator, not 50 times in each function.
- **Separation of Concerns:** Business logic (calculating trade value) stays perfectly isolated from infrastructure logic (checking user authentication or measuring latency).
- **Clean Syntax:** The `@decorator_name` syntax acts as clean metadata tagging.

### The Key Insight

In Python, functions are "First-Class Objects". This means you can pass a function into another function as an argument, just like you would pass an integer or a string.

## 🔍 Functions as First-Class Objects

In [ ]:
# Demonstrating passing functions as arguments

def print_hello():
    print("Hello!")
    
def execute_twice(func):
    # Note we accept 'func' (the object), and then we call it with ()
    print("Executing provided function twice:")
    func()
    func()
    
# Notice we pass `print_hello`, NOT `print_hello()`
execute_twice(print_hello)

## 🏗️ The Anatomy of a Decorator

A standard decorator has three layers:
1. The outer function (takes the original function).
2. The inner wrapper function (adds the `*args, **kwargs` to catch any inputs, executes the pre-code, calls the original function, executes the post-code).
3. Returning the wrapper function (uncalled).

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("--- BEFORE EXECUTION ---")
        result = func(*args, **kwargs) # Call the original
        print("--- AFTER EXECUTION ---")
        return result
    return wrapper

@my_decorator
def say_hello(name):
    print(f"Hello {name}!")

say_hello("Sean")

## 🔄 Manual Wrapping vs '@' Syntactic Sugar

The `@` symbol is literally just a shortcut. It does nothing magical.

In [ ]:
def simple_func():
    print("Standard execution")

# Exactly what the @ symbol does under the hood:
wrapped_func = my_decorator(simple_func)
wrapped_func()

print("\nIs exactly the same as:\n")
print("""
@my_decorator
def simple_func():
    print("Standard execution")
""")

## 🌍 Real-World Example: The `@timer` Decorator

**The Citi Context:** When profiling ETL scripts, we needed to know how long 15 different data transformation functions took to execute. We absolutely could not dirty the pandas logic with `time.time()` boilerplate in 15 different places.

### The Solution
A universal `@timer` decorator placed above any function we wanted to benchmark.

In [ ]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"[PROFILER] {func.__name__} executed in {(end_time - start_time):.4f} seconds")
        return result
    return wrapper

@timer
def heavy_computation():
    time.sleep(1.2) # Simulating an ETL heavy join
    return "Success"

heavy_computation()

## 🎭 Advanced: Decorators with Arguments

What if we want to pass a parameter to the decorator itself? Like `@retry(retries=3)`. We need a *third* layer of nesting.

In [ ]:
def retry(max_retries=3):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt + 1} failed: {e}")
            raise Exception(f"Function failed after {max_retries} retries.")
        return wrapper
    return decorator

test_counter = 0
@retry(max_retries=3)
def unstable_api_call():
    global test_counter
    test_counter += 1
    if test_counter < 3:
        raise ValueError("Network timeout!")
    return "200 OK"

print("Final result:", unstable_api_call())

## 🎤 5 Interview Q&A

### Q1: What does `wraps` from `functools` do?
**Answer:** When you wrap a function, the new wrapper function overwrites the original function's name and docstring (it now literally thinks its name is `wrapper`). `@functools.wraps(func)` is a decorator you put *inside* your decorator to copy the original `__name__` and `__doc__` over to the wrapper, preserving metadata.

---

### Q2: Can you stack multiple decorators?
**Answer:** Yes. If you put `@timer` above `@retry`, they execute bottom-up for the wrapping, and top-down for the calling. The timer will wrap the retry logic, so it will time the sum total of all 3 retries.

---

### Q3: Why does the wrapper need `*args, **kwargs`?
**Answer:** To make the decorator universal. If the wrapper doesn't accept arguments, you can only decorate functions that take zero parameters. `*args` and `**kwargs` catch every possible positional and keyword argument and pass them cleanly into the original function.

---

### Q4: What is a Class Decorator?
**Answer:** The same concept applied to a Class definition instead of a function. You decorate the class, the decorator receives the Class object, and typically modifies it (like adding a `__repr__` method automatically) and returns the modified Class. Python's `@dataclass` is exactly this.

---

### Q5: How is this different from a Context Manager (`with` statement)?
**Answer:** A Context Manager is executed dynamically around a block of code at runtime to manage resources (like opening/closing files). A Decorator is executed exactly once, at import/compile time, when the function is defined, permanently altering how that function completely behaves for the life of the program.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **First-Class Object** | An entity (like a function) that can be passed as an argument or assigned to a variable. | `x = my_func` |
| **Wrapper** | The inner function inside a decorator that actually intercepts the call. | `def wrapper():` |
| **Syntactic Sugar** | Syntax designed to make things easier to read, though it maps to standard logic. | `@timer` |
| **args / kwargs** | Positional arguments (tuple) / Keyword arguments (dict) catching all generic inputs. | `(*args, **kwargs)` |
| **Higher-Order Function**| A function that either takes a function as an arg, or returns a function. | Any decorator! |

## 💼 The Citi Narrative: Access Control

### The Problem
We built a Flask API serving bond pricing data. There were 20 different routing endpoints. Management decided that 5 of these endpoints required "Admin" level Active Directory authentication.

### The Solution (The Bad Way)
Adding `if current_user.role != 'Admin': return 403` to the first two lines of all 5 functions.

### The Solution (The Citi Engineer Way)
I built a `@requires_admin` decorator. It checked the Active Directory token in the request header. If invalid, the wrapper aborted the request. If valid, it executed the endpoint function.

### The Impact
Instead of injecting security logic into mathematical pricing logic, the code remained perfectly pure. We simply added the `@requires_admin` tag above the 5 specific endpoints. Auditing the security of the app became a matter of skimming the tags.

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: The Debugger
# Write a decorator @debug that prints the name of the function and the arguments it was called with.
def debug(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args} kwargs={kwargs}")
        return func(*args, **kwargs)
    return wrapper

@debug
def add(x, y):
    return x + y
    
add(5, 10)

In [ ]:
# EXERCISE 2: Missing Return
# What bug occurs if your `wrapper` forgets to `return func(*args, **kwargs)`?
print("Your decorated function will always return `None`, silently destroying the expected output of your business logic.")

In [ ]:
# EXERCISE 3: functools.wraps
# Demonstrate why @wraps is needed.
from functools import wraps

def my_dec(func):
    @wraps(func) # Uncomment this to fix the issue
    def wrapper(*args):
        return func(*args)
    return wrapper

@my_dec
def multiply(a, b):
    """Multiplies two numbers."""
    return a * b

print("Name:", multiply.__name__) # Would be 'wrapper' without @wraps
print("Doc:", multiply.__doc__)   # Would be None without @wraps

## 🎯 Summary: Key Takeaways

### What You've Learned
1. ✅ **First-Class:** Functions can be passed like variables.
2. ✅ **The Wrapper Pattern:** Outer func accepts target, Inner func does the intercepting, outer returns inner.
3. ✅ **The Goal:** Separation of concerns. Injecting logic (timing, logging, auth, retries) without touching core code.
4. ✅ **Nesting:** You need 3 function layers if your decorator itself accepts parameters.
5. ✅ **functools.wraps:** The required boilerplate to avoid destroying your function's metadata.

### Interview Confidence Checklist
- [ ] Can accurately write a `@timer` decorator from scratch on a whiteboard.
- [ ] Explain exactly what `*args, **kwargs` is doing inside the wrapper.
- [ ] Explain the difference between compile-time applying of decorators vs run-time execution of the wrapper.
- [ ] Recite the Citi "Flask Endpoint Access Control" story.

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Decorators separate scripters from framework engineers. 🚀